In [1]:
#config

In [2]:
import os
from pathlib import Path

# Configuration parameters
ALGORITHM = "simclr"
IMG_SIZE = 224
INIT_LR = 1e-3
TEMPERATURE = 0.5
BATCH_SIZE = 64
PRE_TRAIN_EPOCHS = 20

VAL_STEPS_PER_EPOCH = 20
WEIGHT_DECAY = 5e-4
DIM = 1024
WARMUP_LR = 0.0
WARMUP_STEPS = 0

# Paths
CSV_PATH = "/rsrch5/home/trans_mol_path/cercan/BE_master/1.aneuploid/results/training_250131/sample_list_mda_12.csv"
BASE_PATH = "/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid"
FOUNDATION_MODEL_BASE = '/rsrch5/home/trans_mol_path/cercan/foundationModels/physionet.org/files/medical-ai-research-foundation/1.0.0/'
DATA_PATH = Path('/rsrch5/home/trans_mol_path/cercan/foundationModels/finetuned/')

In [3]:
# utils

In [4]:
import os
import random
import numpy as np
import cv2
import openslide
import tensorflow as tf

def parse_image_mask(file_path, sthresh=30):
    wsi = openslide.OpenSlide(file_path)
    level = wsi.get_best_level_for_downsample(64)
    image_ds = np.array(wsi.read_region((0, 0), level, wsi.level_dimensions[level]).convert('RGB'))
    img_gray = cv2.cvtColor(image_ds, cv2.COLOR_RGB2GRAY)
    img_blur = cv2.GaussianBlur(img_gray, (7, 7), 0)
    _, img_otsu = cv2.threshold(img_blur, sthresh, 220, cv2.THRESH_OTSU + cv2.THRESH_BINARY_INV)
    kernel = np.ones((2, 2), np.uint8)
    mask = cv2.morphologyEx(img_otsu, cv2.MORPH_CLOSE, kernel)

    original_dimensions = wsi.level_dimensions[0]
    mask = cv2.resize(mask, (original_dimensions[0], original_dimensions[1]), interpolation=cv2.INTER_NEAREST)
    image = np.array(wsi.read_region((0, 0), 0, wsi.level_dimensions[0]).convert('RGB'))

    return image, mask

def generate_random_crop_boundary(image_shape, crop_size):
    height, width = image_shape[:2]
    if height < crop_size:
        start_y = 0
        end_y = height
    else:
        max_y = height - crop_size
        start_y = random.randint(0, max_y)
        end_y = start_y + crop_size

    if width < crop_size:
        start_x = 0
        end_x = width
    else:
        max_x = width - crop_size
        start_x = random.randint(0, max_x)
        end_x = start_x + crop_size

    return start_y, start_x, end_y, end_x

def check_tissue_content(mask, crop_start_y, crop_start_x, crop_dim=448, threshold=0.5):
    crop_mask = mask[crop_start_y:crop_start_y + crop_dim, crop_start_x:crop_start_x + crop_dim]
    tissue_pixels = np.sum(crop_mask > 0)
    total_pixels = crop_mask.size
    return (tissue_pixels / total_pixels) >= threshold

def get_crops_bounds(mask, region_dim=1024, crop_dim=448, max_attempts=20):
    threshold = 0.5
    for _ in range(max_attempts):
        crop_start_y, crop_start_x, _, _ = generate_random_crop_boundary((region_dim, region_dim), crop_dim)
        if check_tissue_content(mask, crop_start_y, crop_start_x):
            crop_bounds = (crop_start_y, crop_start_x)
            return crop_bounds
        else:
            threshold -= 0.05
    return None

2025-02-15 00:10:59.048416: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
#augmenter

In [6]:
import tensorflow as tf
import numpy as np
import tensorflow_similarity as tfsim


import albumentations as A

def simsiam_augmenter(img):
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, [224, 224])
    img = img.numpy()

    transform = A.Compose([
        # A.RandomResizedCrop(224, 224, scale=(0.2, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
        A.ToGray(p=0.05),
        A.GaussianBlur(blur_limit=(3, 7), p=0.5),
    ])

    augmented = transform(image=img)['image']
    augmented = tf.convert_to_tensor(augmented, dtype=tf.float32)
    augmented = tf.clip_by_value(augmented, 0.0, 255.0)

    return augmented


# def simsiam_augmenter(img):
#     img = tf.cast(img, tf.float32)
#     img = tf.image.resize(img, [224, 224])
#     img /= 255.0

#     def _jitter_transform(x):
#         return tfsim.augmenters.augmentation_utils.color_jitter.color_jitter_rand(
#             x,
#             np.random.uniform(0.0, 0.4),
#             np.random.uniform(0.0, 0.4),
#             np.random.uniform(0.0, 0.4),
#             np.random.uniform(0.0, 0.1),
#             "multiplicative",
#         )

#     try:
#         img = tfsim.augmenters.augmentation_utils.random_apply.random_apply(_jitter_transform, p=0.8, x=img)
#     except Exception as e:
#         pass

#     def _grascayle_transform(x):
#         return tfsim.augmenters.augmentation_utils.color_jitter.to_grayscale(x)

#     try:
#         img = tfsim.augmenters.augmentation_utils.random_apply.random_apply(_grascayle_transform, p=0.05, x=img)
#     except Exception as e:
#         pass

#     try:
#         img = tfsim.augmenters.augmentation_utils.blur.random_blur(img, height=7, width=7, min_sigma=0.2, max_sigma=1.2, p=0.5)
#     except Exception as e:
#         pass

#     img = tf.image.random_flip_left_right(img)
#     img = tf.image.random_flip_up_down(img)
#     img = img * 255.0
#     img = tf.clip_by_value(img, 0.0, 255.0)

#     return img

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Your CPU supports instructions that this binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2
For maximum performance, you can install NMSLIB from sources 
pip install --no-binary :all: nmslib
/usr/local/lib/python3.8/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.4 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [7]:
#dataloader

In [8]:
import os
import glob
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
import albumentations as A
import numpy as np


def get_train_val_lists(csv_path):
    slide_list = pd.read_csv(csv_path)
    slide_list = slide_list.drop_duplicates(subset='slide_id', keep='first')
    aneuploid_list = slide_list.loc[slide_list['label'] == 'diploid', 'slide_id']
    diploid_list = slide_list.loc[slide_list['label'] == 'diploid', 'slide_id']

    train_aneuploid, val_aneuploid = train_test_split(aneuploid_list, test_size=0.2, random_state=42)
    train_diploid, val_diploid = train_test_split(diploid_list, test_size=0.2, random_state=42)
    train_list = pd.concat([train_aneuploid, train_diploid], ignore_index=True)
    val_list = pd.concat([val_aneuploid, val_diploid], ignore_index=True)

    return train_list, val_list

def get_tif_files(base_path, csv_path):
    train_list, val_list = get_train_val_lists(csv_path)
    tif_files_train = []
    tif_files_val = []
    for slide_id in train_list:
        tif_files_train += glob.glob(f"{base_path}/{slide_id}*.tif")
    for slide_id in val_list:
        tif_files_val += glob.glob(f"{base_path}/{slide_id}*.tif")
    print(f"Number of training biopsies: {len(train_list)}")
    print(f"Number of validation biopsies: {len(val_list)}")

    return tif_files_train, tif_files_val

def create_random_square(width_dim=None,height_dim=None ):
    if width_dim is None:
        width_dim = height_dim
    return A.Compose([
        A.RandomCrop(width=width_dim, height=height_dim, p=1.0, pad_if_needed = True),
    ])
    
def check_tissue_content(mask, threshold =0.5):
    
    tissue_pixels = np.sum(mask > 0)
    total_pixels = mask.size

    return (tissue_pixels  / total_pixels) >= threshold


def get_crops(filename, region_dim=1024, crop_dim=448):
    wsi_img, wsi_mask = parse_image_mask(filename)
    valid_patch = False
    crops = []

    width_dim = min(wsi_img.shape[1], region_dim)
    height_dim = min(wsi_img.shape[0], region_dim)
        
    get_region = create_random_square(width_dim , height_dim)
    get_crop = create_random_square(crop_dim)

    threshold = 0.5
    max_attempts = 10

    while not valid_patch:
        region = get_region(image=wsi_img, mask=wsi_mask)
        region_mask = region['mask']
        region_image = region['image']
        print(filename)
        # print(region_image.shape)
        crop = tfsim.augmenters.augmentation_utils.cropping.crop_and_resize(
        region_image, CIFAR_IMG_SIZE, CIFAR_IMG_SIZE, area_range=area_range
    )

        for _ in range(max_attempts):
            try:
                crop = get_crop(image=region_image, mask=region_mask)
            except RuntimeError as e:
                print(e)
                print('Error in crop '+ {filename})
            crop_test = check_tissue_content(crop['mask'], threshold)
            if crop_test:
                img_crop = tf.convert_to_tensor(crop['image'], dtype=tf.float32)
                crops.append(img_crop)
                if len(crops) == 2:
                    valid_patch = True
                    break
            else:
                threshold -= 0.1

    return crops


    # while not valid_patch:
    #     augmented = transform(image=wsi_img, mask=wsi_mask)
    #     img_crop = augmented['image']
    #     mask_crop = augmented['mask']

    #     tissue_pixels = np.sum(mask_crop > 0)
    #     total_pixels = mask_crop.size
    #     if (tissue_pixels / total_pixels) >= 0.5:
    #         img_crop = tf.convert_to_tensor(img_crop, dtype=tf.float32)
    #         crops.append(img_crop)

    # return crops

# def get_crops(filename, region_dim=1024, crop_dim=448):
#     wsi_img, wsi_mask = parse_image_mask(filename)
#     valid_patch = False
#     crops = []
#     while not valid_patch:
#         region_start_y, region_start_x, end_y, end_x = generate_random_crop_boundary(wsi_mask.shape, region_dim)
#         region_mask = wsi_mask[region_start_y:end_y, region_start_x:end_x].copy()
#         crop_bounds = get_crops_bounds(region_mask, region_dim=region_dim, crop_dim=crop_dim)
#         if crop_bounds is not None:
#             region_img = wsi_img[region_start_y:end_y, region_start_x:end_x].copy()
#             img_crop = region_img[crop_bounds[0]:crop_bounds[0] + crop_dim, crop_bounds[1]:crop_bounds[1] + crop_dim].copy()
#             img_crop = tf.convert_to_tensor(img_crop, dtype=tf.float32)
#             crops.append(img_crop)
#             if len(crops) == 2:
#                 valid_patch = True
#     return crops

def get_views(file_path):
    file_path = file_path.numpy().decode()
    crops = get_crops(file_path)
    view1 = simsiam_augmenter(crops[0])
    view2 = simsiam_augmenter(crops[1])
    view1.set_shape((224, 224, 3))
    view2.set_shape((224, 224, 3))
    return view1, view2

@tf.function
def tf_read_slide(file_path):
    view1, view2 = tf.py_function(get_views, [file_path], [tf.float32, tf.float32])
    view1.set_shape((224, 224, 3))
    view2.set_shape((224, 224, 3))
    return view1, view2

def create_datasets(tif_files_train, tif_files_val, batch_size=32):
    train_ds = tf.data.Dataset.from_tensor_slices(tif_files_train)
    train_ds = train_ds.repeat()
    train_ds = train_ds.shuffle(1024)
    train_ds = train_ds.map(tf_read_slide, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.batch(batch_size)
    train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

    val_ds = tf.data.Dataset.from_tensor_slices(tif_files_val)
    val_ds = val_ds.repeat()
    val_ds = val_ds.shuffle(1024)
    val_ds = val_ds.map(tf_read_slide, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(batch_size)
    val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds

In [9]:
#callbacks

In [10]:
import tensorflow as tf
import tensorflow_similarity.callbacks as tfsim_callbacks

def create_callbacks(log_dir, chkpt_dir):
    tbc = tf.keras.callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=1,
        update_freq=100,
    )
    mcp = tf.keras.callbacks.ModelCheckpoint(
        filepath=chkpt_dir,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
    )
    return [tbc, mcp]

In [11]:
#model

In [13]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [14]:
device = '/gpu:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
print(device)

/gpu:0


In [15]:
tf.config.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'),
 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [25]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import time
import numpy as np
import tensorflow as tf
# import matplotlib.pyplot as plt\
import tensorflow_similarity.losses as tfsim_losses
import tensorflow_addons as tfa

In [17]:
tif_files_train, tif_files_val = get_tif_files(BASE_PATH, CSV_PATH)
train_ds, val_ds = create_datasets(tif_files_train, tif_files_val)
PRE_TRAIN_STEPS_PER_EPOCH = len(tif_files_train) // BATCH_SIZE

Number of training biopsies: 474
Number of validation biopsies: 120


2025-02-15 00:11:09.394452: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38367 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:b7:00.0, compute capability: 8.0


In [22]:
arch='path-50x1-remedis-s'
remedis_model = foundationModel()

In [19]:
contrastive_model = create_contrastive_model(backbone=remedis_model)
loss = tfsim_losses.SimCLRLoss(name=ALGORITHM, temperature=TEMPERATURE)
optimizer = tfa.optimizers.LAMB(learning_rate=INIT_LR)
contrastive_model = compile_model(contrastive_model, optimizer, loss)
contrastive_model.summary()

[Backbone]
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 7, 7, 2048)        23496256  
                                                                 
 average_pooling2d (AverageP  (None, 1, 1, 2048)       0         
 ooling2D)                                                       
                                                                 
 flatten (Flatten)           (None, 2048)              0         
                                                                 
 dense (Dense)               (None, 1024)              2097152   
                                                                 
Total params: 25,593,408
Trainable params: 2,097,152
Non-trainable params: 23,496,256
_________________________________________________________________

[Projector]
Model: "projector"
___________________________________________________________

In [23]:
log_dir = DATA_PATH / arch / "models" / "logs" / f"{loss.name}_{time.time()}"
chkpt_dir = DATA_PATH / arch / "models" / "checkpoints" / f"{loss.name}_{time.time()}"
callbacks = create_callbacks(log_dir, chkpt_dir)


In [24]:
history = contrastive_model.fit(
        train_ds,
        epochs=PRE_TRAIN_EPOCHS,
        steps_per_epoch=PRE_TRAIN_STEPS_PER_EPOCH,
        validation_data=val_ds,
        validation_steps=VAL_STEPS_PER_EPOCH,
        callbacks=callbacks,
        verbose=1,
)

Epoch 1/20


2025-02-15 00:21:42.966586: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [948]
	 [[{{node Placeholder/_0}}]]
2025-02-15 00:21:42.966902: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [948]
	 [[{{node Placeholder/_0}}]]
2025-02-15 00:21:44.227736: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/block4/unit03/StatefulPartitionedCall_gra

/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid/D0721_ROI1.tif
/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid/D0629_ROI0.tif


2025-02-15 00:21:59.120712: W tensorflow/core/framework/op_kernel.cc:1818] INVALID_ARGUMENT: TypeError: '>' not supported between instances of 'NoneType' and 'int'
Traceback (most recent call last):

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 265, in __call__
    return func(device, token, args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 143, in __call__
    outputs = self._call(device, args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 150, in _call
    ret = self._func(*args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/autograph/impl/api.py", line 642, in wrapper
    return func(*args, **kwargs)

  File "/tmp/ipykernel_6366/781165539.py", line 123, in get_views
    crops = get_crops(file_path)

  File "/tmp/ipykernel_6366/781165539.py", line 74, in get_crops
    crop = get_crop(image=region_image, mask=region_mask)

  File 

/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid/D0545_ROI1.tif
/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid/D0868_ROI0.tif


...

2025-02-15 00:22:15.285307: W tensorflow/core/framework/op_kernel.cc:1818] INVALID_ARGUMENT: TypeError: '>' not supported between instances of 'NoneType' and 'int'
Traceback (most recent call last):

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 265, in __call__
    return func(device, token, args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 143, in __call__
    outputs = self._call(device, args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 150, in _call
    ret = self._func(*args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/autograph/impl/api.py", line 642, in wrapper
    return func(*args, **kwargs)

  File "/tmp/ipykernel_6366/781165539.py", line 123, in get_views
    crops = get_crops(file_path)

  File "/tmp/ipykernel_6366/781165539.py", line 74, in get_crops
    crop = get_crop(image=region_image, mask=region_mask)

  File 

/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid/D0973_ROI1.tif


2025-02-15 00:22:21.219275: W tensorflow/core/framework/op_kernel.cc:1818] INVALID_ARGUMENT: TypeError: '>' not supported between instances of 'NoneType' and 'int'
Traceback (most recent call last):

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 265, in __call__
    return func(device, token, args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 143, in __call__
    outputs = self._call(device, args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 150, in _call
    ret = self._func(*args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/autograph/impl/api.py", line 642, in wrapper
    return func(*args, **kwargs)

  File "/tmp/ipykernel_6366/781165539.py", line 123, in get_views
    crops = get_crops(file_path)

  File "/tmp/ipykernel_6366/781165539.py", line 74, in get_crops
    crop = get_crop(image=region_image, mask=region_mask)

  File 

/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid/D0063_ROI1.tif


2025-02-15 00:22:22.808784: W tensorflow/core/framework/op_kernel.cc:1818] INVALID_ARGUMENT: TypeError: '>' not supported between instances of 'NoneType' and 'int'
Traceback (most recent call last):

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 265, in __call__
    return func(device, token, args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 143, in __call__
    outputs = self._call(device, args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 150, in _call
    ret = self._func(*args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/autograph/impl/api.py", line 642, in wrapper
    return func(*args, **kwargs)

  File "/tmp/ipykernel_6366/781165539.py", line 123, in get_views
    crops = get_crops(file_path)

  File "/tmp/ipykernel_6366/781165539.py", line 74, in get_crops
    crop = get_crop(image=region_image, mask=region_mask)

  File 